<a href="https://colab.research.google.com/github/neelkumar01/Effect-of-model-size-and-accuracy-with-chain-of-thought-reasoning/blob/main/medium_language_model_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### The Effect of Model Size on Accuracy with Chain of Thought Reasoning Under Inference Time Compute Scaling

### Experiment 1 on tiny language models

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch
import re
from tqdm import tqdm

#### Test data (Summary)

GSM8K (Grade School Math 8K) is a dataset of 8.5K high quality linguistically diverse grade school math word problems. The dataset was created to support the task of question answering on basic mathematical problems that require multi-step reasoning.

- These problems take between 2 and 8 steps to solve.

- Solutions primarily involve performing a sequence of elementary calculations using basic arithmetic operations (+ − ×÷) to reach the final answer.

- A bright middle school student should be able to solve every problem: from the paper, "Problems require no concepts beyond the level of early Algebra, and the vast majority of problems can be solved without explicitly defining a variable."

- Solutions are provided in natural language, as opposed to pure math expressions. From the paper: "We believe this is the most generally useful data format, and we expect it to shed light on the properties of large language models’ internal monologues""

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test[:50]")

In [ ]:
gsm8k[0]["question"]

In [ ]:
gsm8k[0]["answer"]

#### Few shot prompts

In [ ]:
few_shots = """Q: Emily has 3 apples. Her friend gives her 2 more. How many apples does Emily have now?
A: Emily starts with 3 apples. Her friend gives her 2 more. So, 3 + 2 = 5. The answer is 5.

Q: A pen costs 2 dollars. John buys 4 pens. How much does he pay?
A: Each pen costs 2 dollars. John buys 4 pens. So, 2 × 4 = 8. The answer is 8.

Q: Jake read 5 pages on Monday and 7 pages on Tuesday. How many pages did he read in total?
A: Jake read 5 pages on Monday and 7 on Tuesday. So, 5 + 7 = 12. The answer is 12.

Q: A bakery bakes 48 cookies and packs them equally into 6 boxes. They sell 3 boxes. How many cookies are left?
A: First, find cookies per box: 48 ÷ 6 = 8 cookies per box. They sell 3 boxes, so cookies sold = 3 × 8 = 24. Cookies remaining = 48 - 24 = 24. The answer is 24.

Q: Maria earns 15 dollars per hour. She works 6 hours on Saturday and 4 hours on Sunday. How much does she earn in total?
A: Saturday earnings = 15 × 6 = 90 dollars. Sunday earnings = 15 × 4 = 60 dollars. Total earnings = 90 + 60 = 150. The answer is 150.
"""

#### Extracting final answer as a single numerical value

In [ ]:
def extract_final_answer(text):
    text = text.replace(",", "")
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]
    return None

#### Loading specific model - seq2seq or casual LM model

In [ ]:
def load(model_id, use_4bit = True):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(model_id)

    if config.is_encoder_decoder:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_id,
            dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)

        model_type = "seq2seq"

    else:
        if device == "cuda" and use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.float16 if device == "cuda" else torch.float32
            ).to(device)

        model_type = "causal"

    model.eval()
    return tokenizer, model, model_type, device

In [ ]:
def evaluate(model_id, batch_size=8):
    print(f"Evaluating {model_id}")

    tokenizer, model, model_type, device = load(model_id)

    prompts = []
    answers = []

    for qaPair in tqdm(gsm8k, desc="Preparing prompts"):
        question = qaPair["question"]
        answer = qaPair["answer"].split("####")[-1].strip().replace(",", "")

        prompt = few_shots + f"\n\nQ: {question}\nA:"
        prompts.append(prompt)
        answers.append(answer)

    outputs = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating answers"):
        batch_prompts = prompts[i:i + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        if model_type == "seq2seq":
            inputs = inputs.to(device)

        else:
            inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        if model_type == "causal":
            input_length = inputs["input_ids"].shape[1]
            generated_ids = generated_ids[:, input_length:]

        batch_texts = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        outputs.extend(batch_texts)

    correct = 0

    for generated_text, answer in tqdm(
        zip(outputs, answers),
        total=len(answers),
        desc="Checking accuracy"
    ):
        pred = extract_final_answer(generated_text)

        if pred == answer:
            correct += 1

    total = len(gsm8k)
    accuracy = correct / total

    print(f"Accuracy: {accuracy}")
    return accuracy

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"
model_ids = {
    "Phi-3 Mini 3.8B" : "microsoft/Phi-3-mini-4k-instruct",
    "Qwen2.5 0.5B" : "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen2.5 math 1.5B" : "Qwen/Qwen2.5-Math-1.5B-Instruct"

}

results = []

for label, model_id in model_ids.items():
    acc = evaluate(model_id, batch_size=8)
    results.append({
        "Model": label,
        "Accuracy": acc
    })

fig = px.bar(
    results,
    x="Model",
    y="Accuracy",
    text="Accuracy",
    title="Few-Shot CoT Reasoning Accuracy vs Model Size (GSM8K)",
)

fig.update_traces(
    texttemplate="%{text}",
    textposition="outside"
)

fig.update_layout(
    yaxis=dict(
        title="Accuracy",
        range=[0, 1]
    ),
    xaxis_title="Model",
    template="plotly_white"
)

fig.show()